In [1]:
import numpy as np 
import pandas as pd
import os

In [2]:
link='/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/ml-1m'
rating_data=os.path.join(link,'ratings.dat')
movie_data=os.path.join(link,'movies.dat')
user_data=os.path.join(link,'users.dat')

ratings=pd.read_csv(
  rating_data,
  sep='::',
  engine='python',
  names=['user','item','rating','timestamp'],
  encoding='latin-1'   
)

movies=pd.read_csv(
    movie_data, 
    sep='::',
    engine='python',
    names=['item', 'title', 'genres'],
    encoding='latin-1'
)
users = pd.read_csv(
    user_data, 
    sep='::',
    engine='python',
    names=['user', 'gender', 'age', 'occupation', 'zipcode'],
    encoding='latin-1'
)

In [3]:
print(ratings.head())
print(movies.head())
print(users.head())

   user  item  rating  timestamp
0     1  1193       5  978300760
1     1   661       3  978302109
2     1   914       3  978301968
3     1  3408       4  978300275
4     1  2355       5  978824291
   item                               title                        genres
0     1                    Toy Story (1995)   Animation|Children's|Comedy
1     2                      Jumanji (1995)  Adventure|Children's|Fantasy
2     3             Grumpier Old Men (1995)                Comedy|Romance
3     4            Waiting to Exhale (1995)                  Comedy|Drama
4     5  Father of the Bride Part II (1995)                        Comedy
   user gender  age  occupation zipcode
0     1      F    1          10   48067
1     2      M   56          16   70072
2     3      M   25          15   55117
3     4      M   45           7   02460
4     5      M   25          20   55455


In [16]:
import cornac
from cornac.eval_methods import RatioSplit
from cornac.models import SVD, ItemKNN, UserKNN, MostPop, NeuMF, MF
from cornac.metrics import RMSE, NDCG, Precision, MAE

In [ ]:
# Cornac wymaga listy krotek w układzie [(user_id, item_id, rating), ...]
data = [tuple(x) for x in ratings[['user', 'item', 'rating']].to_numpy()]

In [ ]:
 #  w rating_threshold oceny > wartości uznawane są za pozytywne, jest to ważne przy metrykach rankingowych
rs = RatioSplit( data=data, test_size=0.2, rating_threshold=1.0, seed=42, verbose=True)

svd = SVD(
    k=10, # l. ukrytych cech (wymiarów), odpowiednik n_factors w Surprise
    max_iter=30, 
    learning_rate=0.01, 
    lambda_reg=0.015, 
    verbose=True
)

rating_threshold = 1.0
exclude_unknowns = True
---
Training data:
Number of users = 6040
Number of items = 3675
Number of ratings = 800167
Max rating = 5.0
Min rating = 1.0
Global mean = 3.6
---
Test data:
Number of users = 6040
Number of items = 3675
Number of ratings = 200006
Number of unknown users = 0
Number of unknown items = 0
---
Total users = 6040
Total items = 3675


In [ ]:
experimentSVD = cornac.Experiment(
    eval_method=rs,
    models=[svd],
    metrics=[RMSE(), MAE()]
)

experimentSVD.run()

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\cornac\models\recommender.py:322: UserWarning: Model is already fitted. Re-fitting will overwrite the previous model.
  warnings.warn(



[SVD] Training started!


100%|██████████| 30/30 [00:00<00:00, 54.43it/s, loss=233186.23]


Optimization finished!

[SVD] Evaluation started!


Rating: 100%|██████████| 200006/200006 [00:02<00:00, 98743.99it/s] 



TEST:
...
    |    MAE |   RMSE | Train (s) | Test (s)
--- + ------ + ------ + --------- + --------
SVD | 0.7017 | 0.8680 |    0.5588 |   2.9765



In [ ]:
iknn = ItemKNN(
    k=20, # l. sąsiadów, czyli ile podobnych filmów jest branych pod uwagę
    similarity='cosine', # miara podobieństwa - cosinus kąta, może też być pearson lub msd
    verbose=True #wyświetlanie
)

In [18]:
experimentIknn = cornac.Experiment(
    eval_method=rs,
    models=[iknn],
    metrics=[RMSE(), MAE()]
)

experimentIknn.run()


[ItemKNN] Training started!


100%|██████████| 3675/3675 [00:01<00:00, 2841.20it/s]



[ItemKNN] Evaluation started!


Rating: 100%|██████████| 200006/200006 [00:29<00:00, 6776.78it/s]



TEST:
...
        |    MAE |   RMSE | Train (s) | Test (s)
------- + ------ + ------ + --------- + --------
ItemKNN | 0.7946 | 0.9804 |    2.1478 |  30.9475



In [ ]:
uknn = UserKNN(
    k=20, # l. sąsiadów, czyli ile podobnych filmów jest branych pod uwagę
    similarity='cosine', # miara podobieństwa - cosinus kąta, może też być pearson lub msd
    mean_centered=True, # niweluje różnice między użytkownikami, którzy dają same oceny 5 i takimi co 3
    verbose=True #wyświetlanie
)

In [20]:
experimentUknn = cornac.Experiment(
    eval_method=rs,
    models=[uknn],
    metrics=[RMSE(), MAE()]
)

experimentUknn.run()


[UserKNN] Training started!


100%|██████████| 6040/6040 [00:02<00:00, 2321.08it/s]



[UserKNN] Evaluation started!


Rating: 100%|██████████| 200006/200006 [00:32<00:00, 6157.05it/s]



TEST:
...
        |    MAE |   RMSE | Train (s) | Test (s)
------- + ------ + ------ + --------- + --------
UserKNN | 0.7561 | 0.9281 |    4.4063 |  33.9235



In [ ]:
experimentMostpop = cornac.Experiment(
    eval_method=rs,
    models=[MostPop()],
    metrics=[NDCG(k=5),Precision(k=5)]
)

experimentMostpop.run()


[MostPop] Training started!

[MostPop] Evaluation started!


Ranking: 100%|██████████| 6033/6033 [00:03<00:00, 1880.13it/s]



TEST:
...
        | NDCG@5 | Precision@5 | Train (s) | Test (s)
------- + ------ + ----------- + --------- + --------
MostPop | 0.2222 |      0.2114 |    0.0004 |   3.2380



In [21]:
mf = MF(
    k=10,                
    max_iter=30,         
    learning_rate=0.01, 
    lambda_reg=0.015,   
    seed=42,            
    verbose=True
)

In [22]:
experimentMf = cornac.Experiment(
    eval_method=rs,
    models=[mf],
    metrics=[RMSE(), MAE()]
)
experimentMf.run()


[MF] Training started!


100%|██████████| 30/30 [00:01<00:00, 20.83it/s, loss=243056.94]


Optimization finished!

[MF] Evaluation started!


Rating: 100%|██████████| 200006/200006 [00:02<00:00, 90843.92it/s]



TEST:
...
   |    MAE |   RMSE | Train (s) | Test (s)
-- + ------ + ------ + --------- + --------
MF | 0.6930 | 0.8532 |    1.5071 |   3.2304



In [22]:
import tensorflow as tf

In [ ]:
# sieć neuronowa
neumf = NeuMF(
    num_factors=32, #  wymiar warstwy GMF [Generalized Matrix Factorization] (embedding size)
    layers=[64, 32, 16, 8],     # struktura sieci MLP (l. neuronów w kolejnych warstwach)
    act_fn='relu',    # funkcja aktywacji, może być też tanh
    learner='adam', 
    num_epochs=20, 
    batch_size=256,
    lr=0.001,   #learning rate
    seed=42,
    verbose=True
)

In [25]:
experimentNeumf = cornac.Experiment(
    eval_method=rs,
    models=[neumf],
    metrics=[RMSE(), Precision(k=5), NDCG(k=5)]
)

experimentNeumf.run() 
# w wersji num_epochs=10 i metrics=[RMSE()] experyment trwał około 4,5 h

#RMSE 2.8833
# NDCG@-1 0.5600, Precision@-1 0.0099    Train 9887.5230      Test (s) 3201.0840


[NeuMF] Training started!


c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\cornac\models\recommender.py:322: UserWarning: Model is already fitted. Re-fitting will overwrite the previous model.
  warnings.warn(
100%|██████████| 20/20 [3:43:50<00:00, 671.52s/it, loss=0.214]  



[NeuMF] Evaluation started!


Ranking: 100%|██████████| 6033/6033 [02:01<00:00, 49.73it/s]


TEST:
...
      |   RMSE | NDCG@5 | Precision@5 |  Train (s) |  Test (s)
----- + ------ + ------ + ----------- + ---------- + ---------
NeuMF | 2.8833 | 0.3336 |      0.3218 | 13430.8225 | 5427.9322

